# Strands Agents com AgentCore Memory (Memória de longo prazo) - Sobrescrita de Estratégia Personalizada

## Introdução

Este tutorial demonstra como construir um **agente inteligente de suporte ao cliente** usando Strands Agents integrado com AgentCore Memory via hooks utilizando **MemoryManager** com **estratégias personalizadas** e **MemorySessionManager**. Vamos focar na memória de longo prazo para histórico de interações com clientes, lembrando detalhes de compras e fornecendo suporte personalizado baseado em conversas anteriores e preferências do usuário.

**NOTA: Esta abordagem usa estratégias personalizadas (CustomSemanticStrategy, CustomUserPreferenceStrategy) que REQUEREM uma IAM execution role e permitem especificar modelos personalizados para extração e consolidação.**

### Detalhes do Tutorial

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo de tutorial    | Conversacional de longo prazo                                                    |
| Tipo de agente      | Suporte ao Cliente                                                               |
| Framework de agentes| Strands Agents                                                                   |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                      |
| Componentes do tutorial | Extração de Memória Semântica e de Preferências do Usuário do AgentCore (Personalizada com sobrescrita de modelo), Hooks para armazenamento e recuperação de Memória |
| Complexidade do exemplo | Avançado                                                                     |

Você aprenderá a:
- Configurar o AgentCore Memory com estratégias personalizadas de longo prazo usando MemoryManager
- Configurar modelos personalizados para extração e consolidação
- Criar IAM execution role para invocação de modelos com estratégia personalizada
- Criar hooks de memória para armazenamento e recuperação automáticos com MemorySessionManager
- Construir um agente de suporte ao cliente com memória persistente
- Lidar com problemas de clientes usando contexto de interações anteriores

### Contexto do Cenário
Neste exemplo, vamos construir um **Caso de Uso de Suporte ao Cliente**. O agente lembrará do contexto do cliente, incluindo histórico de pedidos, preferências e problemas anteriores, permitindo um suporte mais personalizado e eficaz. Conversas com clientes são armazenadas automaticamente usando hooks de memória, garantindo que detalhes importantes nunca sejam perdidos. Ao empregar múltiplas estratégias de memória, como semântica e preferências do usuário, o agente pode capturar uma ampla gama de informações relevantes. Essa configuração permite que o agente resolva problemas com total conhecimento do histórico e preferências do cliente. Além disso, o agente é integrado com capacidades de busca na web, facilitando o fornecimento de informações atualizadas sobre produtos e orientações de solução de problemas conforme necessário.

## Arquitetura

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>


## Pré-requisitos

Para executar este tutorial você precisará de:
- Python 3.10+
- Credenciais AWS com permissões do Amazon Bedrock AgentCore Memory
- SDK do Amazon Bedrock AgentCore com suporte ao MemoryManager

## 📊 Escolhendo Entre Estratégias Built-in vs Personalizadas

O AgentCore Memory oferece duas abordagens para extração e consolidação de memória. Este notebook demonstra a abordagem de **Sobrescrita de Estratégia Personalizada**.

### Sobrescrita de Estratégia Personalizada (Este Notebook)

**Quando usar:**
- ✅ Precisa especificar modelos Bedrock personalizados para extração/consolidação
- ✅ Controle refinado sobre o comportamento do modelo
- ✅ Prompts personalizados para extração e consolidação
- ✅ Casos de uso avançados que requerem capacidades específicas do modelo
- ✅ Requisitos de conformidade para versões específicas de modelo

**Características principais:**
- Usa as classes `CustomSemanticStrategy` e `CustomUserPreferenceStrategy`
- Requer IAM execution role para invocação de modelos
- Permite especificação de `ExtractionConfig` e `ConsolidationConfig`
- Configuração mais complexa, porém maior controle
- Útil para requisitos especializados

**Exemplo:**
```python
strategies = [
    CustomSemanticStrategy(
        name="CustomerSupportSemantic",
        extraction_config=ExtractionConfig(
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
            append_to_prompt="Extract factual information..."
        ),
        consolidation_config=ConsolidationConfig(
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
            append_to_prompt="Consolidate semantic insights..."
        ),
        namespaces=["support/customer/{actorId}/semantic/"]
    )
]

memory = memory_manager.get_or_create_memory(
    name="CustomerSupportMemory",
    strategies=strategies,
    memory_execution_role_arn=MEMORY_EXECUTION_ROLE_ARN  # Required!
)
```

### Estratégias Built-in (Veja `customer-support-inbuilt-strategy.ipynb`)

**Quando usar:**
- ✅ Configuração rápida e prototipagem
- ✅ Necessidades padrão de extração de memória
- ✅ Sem necessidade de seleção personalizada de modelo
- ✅ Configuração IAM simplificada (sem necessidade de execution role)
- ✅ Cargas de trabalho em produção com modelos padrão do AgentCore

**Características principais:**
- Usa as classes `SemanticStrategy` e `UserPreferenceStrategy`
- O AgentCore Memory seleciona e gerencia modelos automaticamente
- Não requer IAM execution role
- Configuração mais simples com menos parâmetros
- Ideal para a maioria dos casos de uso

**Exemplo:**
```python
strategies = [
    SemanticStrategy(
        name="CustomerSupportSemantic",
        description="Stores facts from conversations",
        namespaces=["support/customer/{actorId}/semantic/"]
    )
]

memory = memory_manager.get_or_create_memory(
    name="CustomerSupportMemory",
    strategies=strategies
    # No memory_execution_role_arn needed!
)
```

### Tabela Comparativa Rápida

| Recurso | Estratégias Built-in | Sobrescrita de Estratégia Personalizada |
|---------|---------------------|-------------------------|
| **Complexidade de Configuração** | Simples | Avançado |
| **IAM Role Necessária** | ❌ Não | ✅ Sim |
| **Seleção de Modelo** | Automática (gerenciada pelo AgentCore) | Manual (você especifica) |
| **Prompts Personalizados** | ❌ Não | ✅ Sim |
| **Configuração** | Mínima | Detalhada |
| **Caso de Uso** | Extração de memória padrão | Requisitos de modelo personalizados |
| **Recomendado Para** | Maioria das aplicações | Necessidades especializadas |

---

**💡 Recomendação:** Comece com estratégias built-in (`customer-support-inbuilt-strategy.ipynb`) para a maioria dos casos de uso. Use esta abordagem de sobrescrita de estratégia personalizada somente se você tiver requisitos específicos de modelo ou precisar de controle granular sobre o comportamento de extração/consolidação.

## Passo 1: Instalar Dependências e Configuração
Vamos começar importando todas as bibliotecas necessárias e definindo os clientes para fazer este notebook funcionar.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
import json
from typing import Dict, List
from datetime import datetime
from botocore.exceptions import ClientError

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("customer-support")

# Import required modules for Strands Agent
from strands import Agent, tool
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent
from ddgs import DDGS

# Import memory management modules
from bedrock_agentcore_starter_toolkit.operations.memory.manager import Memory, MemoryManager
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies import (
    SemanticStrategy, SummaryStrategy, CustomSemanticStrategy, CustomSummaryStrategy, CustomUserPreferenceStrategy,
    ExtractionConfig, ConsolidationConfig
)
from bedrock_agentcore.memory.constants import (
    BlobMessage, ConversationalMessage, MessageRole, RetrievalConfig
)
from bedrock_agentcore.memory.models import (
    StringValue, EventMetadataFilter, LeftExpression, RightExpression, OperatorType, MemoryRecord
)
from bedrock_agentcore.memory.session import Actor, MemorySession, MemorySessionManager

# Define message role constants
USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT

logger.info("✅ All imports loaded successfully")


In [ ]:
# Configuration - Replace with the correct values
REGION = "us-east-1"  
CUSTOMER_ID = "customer_001"
SESSION_ID = f"support_{datetime.now().strftime('%Y%m%d%H%M%S')}"

# Import boto3 for IAM role creation
import boto3
import json as json_module
from botocore.exceptions import ClientError

logger.info(f"✅ Configuration loaded")
logger.info(f"   Region: {REGION}")
logger.info(f"   Customer ID: {CUSTOMER_ID}")
logger.info(f"   Session ID: {SESSION_ID}")

## Passo 1.1: Criar IAM Role para Estratégias de Memória Personalizadas

Estratégias de memória personalizadas requerem uma execution role que permite ao AgentCore Memory invocar modelos Bedrock para extração e consolidação. Esta role é necessária ao usar `CustomSemanticStrategy` ou `CustomUserPreferenceStrategy`.

In [ ]:
# Create IAM role for AgentCore Memory custom strategies
def create_memory_execution_role():
    """Create IAM role for AgentCore Memory custom strategies with required permissions"""
    iam_client = boto3.client('iam', region_name=REGION)
    
    # Get current AWS account ID
    sts_client = boto3.client('sts', region_name=REGION)
    account_id = sts_client.get_caller_identity()['Account']
    
    role_name = "AgentCoreMemoryExecutionRole"
    role_arn = f"arn:aws:iam::{account_id}:role/{role_name}"
    
    # Trust policy for AgentCore Memory service
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "",
                "Effect": "Allow",
                "Principal": {
                    "Service": [
                        "bedrock-agentcore.amazonaws.com"
                    ]
                },
                "Action": "sts:AssumeRole",
                "Condition": {
                    "StringEquals": {
                        "aws:SourceAccount": account_id
                    },
                    "ArnLike": {
                        "aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:*"
                    }
                }
            }
        ]
    }
    
    # Permissions policy for Bedrock model invocation
    permissions_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "bedrock:InvokeModel",
                    "bedrock:InvokeModelWithResponseStream"
                ],
                "Resource": [
                    "arn:aws:bedrock:*::foundation-model/*",
                    "arn:aws:bedrock:*:*:inference-profile/*"
                ],
                "Condition": {
                    "StringEquals": {
                        "aws:ResourceAccount": account_id
                    }
                }
            }
        ]
    }
    
    try:
        # Check if role already exists
        try:
            existing_role = iam_client.get_role(RoleName=role_name)
            logger.info(f"✅ IAM role already exists: {role_arn}")
            return role_arn
        except ClientError as e:
            if e.response['Error']['Code'] != 'NoSuchEntity':
                raise
        
        # Create the role
        logger.info(f"Creating IAM role: {role_name}")
        iam_client.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json_module.dumps(trust_policy),
            Description="Execution role for AgentCore Memory custom strategies",
            Tags=[
                {
                    'Key': 'Purpose',
                    'Value': 'AgentCoreMemory'
                }
            ]
        )
        
        # Attach the permissions policy
        policy_name = "AgentCoreMemoryBedrockAccess"
        iam_client.put_role_policy(
            RoleName=role_name,
            PolicyName=policy_name,
            PolicyDocument=json_module.dumps(permissions_policy)
        )
        
        logger.info(f"✅ Successfully created IAM role: {role_arn}")
        logger.info(f"   - Trust policy: AgentCore Memory service can assume this role")
        logger.info(f"   - Permissions: bedrock:InvokeModel and bedrock:InvokeModelWithResponseStream")
        
        return role_arn
        
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == 'AccessDenied':
            logger.error("❌ Access denied creating IAM role. Please ensure you have IAM permissions:")
            logger.error("   - iam:CreateRole")
            logger.error("   - iam:PutRolePolicy")
            logger.error("   - iam:GetRole")
        else:
            logger.error(f"❌ Failed to create IAM role: {e}")
        raise
    except Exception as e:
        logger.error(f"❌ Unexpected error creating IAM role: {e}")
        raise

import time

# Create the execution role
try:
    MEMORY_EXECUTION_ROLE_ARN = create_memory_execution_role()
    logger.info(f"✅ Memory execution role ready: {MEMORY_EXECUTION_ROLE_ARN}")
    # Wait for IAM role propagation - newly created roles take time to propagate across AWS
    logger.info("⏳ Waiting 15 seconds for IAM role propagation...")
    time.sleep(15)
    logger.info("✅ IAM role propagation wait complete")
except Exception as e:
    logger.error(f"❌ Failed to create memory execution role: {e}")
    raise

## Passo 2: Criar Recurso de Memória para Suporte ao Cliente

Para suporte ao cliente, usaremos múltiplas estratégias de memória:
- **CustomUserPreferenceStrategy**: Captura preferências e comportamento do cliente
- **CustomSemanticStrategy**: Armazena fatos sobre pedidos e informações de produtos

**IMPORTANTE**: Estratégias personalizadas requerem uma IAM execution role que permite ao AgentCore Memory invocar modelos Bedrock. Criamos esta role no Passo 1.1 acima.

In [ ]:
# Initialize Memory Manager 
memory_manager = MemoryManager(region_name=REGION)
memory_name = "CustomerSupportLongTermMemory"

# Test basic memory manager initialization and connection
logger.info(f"✅ MemoryManager initialized for region: {REGION}")
logger.info(f"Memory manager type: {type(memory_manager)}")

# Define memory strategies using typed classes
strategies = [
    CustomUserPreferenceStrategy(
        name="CustomerPreferences",
        description="Captures customer preferences and behavior",
        extraction_config=ExtractionConfig(
            append_to_prompt="Extract customer preferences and behavior patterns",
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0"
        ),
        consolidation_config=ConsolidationConfig(
            append_to_prompt="Consolidate customer preferences",
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0"
        ),
        namespaces=["support/customer/{actorId}/preferences/"]
    ),
    CustomSemanticStrategy(
        name="CustomerSupportSemantic",
        description="Stores facts from conversations",
        extraction_config=ExtractionConfig(
            append_to_prompt="Extract factual information from customer support conversations",
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0"
        ),
        consolidation_config=ConsolidationConfig(
            append_to_prompt="Consolidate semantic insights from support interactions",
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0"
        ),
        namespaces=["support/customer/{actorId}/semantic/"]
    )
]

# Validate strategy configuration
logger.info(f"✅ Configured {len(strategies)} memory strategies:")
for i, strategy in enumerate(strategies, 1):
    logger.info(f"  {i}. {strategy.name} ({type(strategy).__name__})")
    logger.info(f"     Description: {strategy.description}")
    logger.info(f"     Namespaces: {strategy.namespaces}")
    if hasattr(strategy, 'extraction_config') and strategy.extraction_config:
        logger.info(f"     Extraction Model: {strategy.extraction_config.model_id}")
    if hasattr(strategy, 'consolidation_config') and strategy.consolidation_config:
        logger.info(f"     Consolidation Model: {strategy.consolidation_config.model_id}"),

# Create memory resource using MemoryManager
logger.info(f"Creating memory '{memory_name}' with {len(strategies)} strategies...")

try:
    memory = memory_manager.get_or_create_memory(
        name=memory_name,
        strategies=strategies,         # Pass typed strategy objects
        description="Memory for customer support agent",
        event_expiry_days=90,          # Memories expire after 90 days
        memory_execution_role_arn=MEMORY_EXECUTION_ROLE_ARN,  # Required for Custom strategies
    )
    memory_id = memory.id
    logger.info(f"✅ Successfully created/retrieved memory with MemoryManager:")
    logger.info(f"   Memory ID: {memory_id}")
    logger.info(f"   Memory Name: {memory.name}")
    logger.info(f"   Memory Status: {memory.status}")
    
except Exception as e:
    # Handle any errors during memory creation with enhanced error reporting
    logger.error(f"❌ Memory creation failed: {e}")
    logger.error(f"Error type: {type(e).__name__}")
    import traceback
    traceback.print_exc()
    
    # Cleanup on error - delete the memory if it was partially created
    if 'memory_id' in locals():
        try:
            logger.info(f"Attempting cleanup of partially created memory: {memory_id}")
            memory_manager.delete_memory(memory_id)
            logger.info(f"✅ Successfully cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"❌ Failed to clean up memory: {cleanup_error}")
    
    # Re-raise the original exception
    raise

In [ ]:
# Verify memory manager connection and display existing memories
try:
    existing_memories = memory_manager.list_memories()
    logger.info(f"✅ Memory manager connection successful. Found {len(existing_memories)} existing memories")
    for mem in existing_memories:
        logger.info(f"   - {mem.name} ({mem.id}) - Status: {mem.status}")
except Exception as e:
    logger.error(f"❌ Memory manager test failed: {e}")
    raise

Vamos confirmar se nossa memória contém as estratégias que atribuímos

In [ ]:
# Display memory information using MemoryManager
print(f"Memory ID: {memory.id}")
print(f"Memory Name: {memory.name}")
print(f"Memory Description: {memory.description}")
print(f"Memory Status: {memory.status}")
print(f"Number of strategies: {len(strategies)}")
for i, strategy in enumerate(strategies, 1):
    print(f"  {i}. {strategy.name}: {strategy.description}")

## Passo 3: Criar Ferramentas do Agente

In [ ]:
from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS

@tool
def web_search(query: str, max_results: int = 3) -> str:
    """Search the web for product information, troubleshooting guides, or support articles.
    
    Args:
        query: Search query for product info or troubleshooting
        max_results: Maximum number of results to return
    
    Returns:
        Search results with titles and snippets
    """
    try:
        results = DDGS().text(query, region="us-en", max_results=max_results)
        if not results:
            return "No search results found."
        
        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(f"{i}. {result.get('title', 'No title')}\n   {result.get('body', 'No description')}")
        
        return "\n".join(formatted_results)
    except RatelimitException:
        return "Rate limit reached: Please try again after a short delay."
    except DDGSException as d:
        return f"Search Error: {d}"
    except Exception as e:
        return f"Search error: {str(e)}"

logger.info("✅ Web search tool ready")

@tool
def check_order_status(order_number: str) -> str:
    """Check the status of a customer order.
    
    Args:
        order_number: The order number to check
    
    Returns:
        Order status information
    """
    # Simulate order lookup
    mock_orders = {
        "123456": "iPhone 15 Pro - Delivered on June 5, 2025",
        "654321": "Sennheiser Headphones - Delivered on June 25, 2025, 1-year warranty active",
        "789012": "Samsung Galaxy S23 - In transit, expected delivery on July 1, 2025",
    }
    
    return mock_orders.get(order_number, f"Order {order_number} not found. Please verify the order number.")

logger.info("✅ Check Order Status tool ready")

## Passo 4: Inicializar o Gerenciador de Sessão

**NOVO: Esta seção apresenta o MemorySessionManager para operações de Memória baseadas em sessão.**

In [ ]:
# Initialize the session memory manager
session_manager: MemorySessionManager = MemorySessionManager(memory_id=memory.id, region_name=REGION)

# Create a memory session for the specific customer
customer_session: MemorySession = session_manager.create_memory_session(
    actor_id=CUSTOMER_ID, 
    session_id=SESSION_ID
)

logger.info(f"✅ Session manager initialized for memory: {memory.id}")
logger.info(f"✅ Customer session created for actor: {CUSTOMER_ID}")
logger.info(f"   Session type: {type(customer_session)}")
logger.info(f"   Actor object: {customer_session.get_actor()}")

## Passo 5: Criar Hook Provider de Memória para Suporte ao Cliente
Hooks são funções especiais que executam em pontos específicos do ciclo de vida de execução de um agente. Nosso hook provider personalizado gerenciará automaticamente o contexto de suporte ao cliente:
- **Salvando interações de suporte** após cada resposta usando métodos baseados em sessão
- **Recuperando e injetando contexto relevante** de pedidos e preferências anteriores ao processar novas consultas.


In [ ]:
class CustomerSupportMemoryHooks(HookProvider):
    """Memory hooks for customer support agent - ENHANCED with MemorySession"""
    
    def __init__(self, customer_session: MemorySession):
        # Accept MemorySession directly 
        self.customer_session = customer_session
        
        # Define retrieval configuration for different memory types
        self.retrieval_config = {
            "support/customer/{actorId}/preferences/": RetrievalConfig(top_k=3, relevance_score=0.3),
            "support/customer/{actorId}/semantic/": RetrievalConfig(top_k=5, relevance_score=0.2)
        }
    
    def retrieve_customer_context(self, event: MessageAddedEvent):
        """Retrieve customer context before processing support query using MemorySession"""
        messages = event.agent.messages
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            
            try:
                # Use MemorySession for context retrieval
                relevant_memories = []
                
                # Search across different memory namespaces using MemorySession
                for namespace_template, config in self.retrieval_config.items():
                    # Resolve namespace template with actual actor ID from session
                    resolved_namespace = namespace_template.format(
                        actorId=self.customer_session._actor_id
                    )
                    
                    # Use MemorySession API (no need to pass actor_id/session_id)
                    memories = self.customer_session.search_long_term_memories(
                        query=user_query,
                        namespace_prefix=resolved_namespace,
                        top_k=config.top_k
                    )
                    
                    # Filter by relevance score
                    filtered_memories = [
                        memory for memory in memories
                        if memory.get("score", 0) >= config.relevance_score
                    ]
                    
                    relevant_memories.extend(filtered_memories)
                    logger.info(f"Found {len(filtered_memories)} relevant memories in {resolved_namespace} (filtered from {len(memories)} total)")
                
                # Inject context into agent's system prompt if memories found
                if relevant_memories:
                    context_text = self._format_context(relevant_memories)
                    original_prompt = event.agent.system_prompt
                    enhanced_prompt = f"{original_prompt}\n\nCustomer Context:\n{context_text}"
                    event.agent.system_prompt = enhanced_prompt
                    logger.info(f"✅ Injected {len(relevant_memories)} memories into agent context")
                    
            except Exception as e:
                logger.error(f"Failed to retrieve customer context: {e}")
    
    def _format_context(self, memories: List[MemoryRecord]) -> str:
        """Format retrieved memories for agent context"""
        context_lines = []
        for i, memory in enumerate(memories[:5], 1):  # Limit to top 5
            content = memory.get('content', {}).get('text', 'No content available')
            score = memory.get('score', 0)
            context_lines.append(f"{i}. (Score: {score:.2f}) {content[:200]}...")
        
        return "\n".join(context_lines)
    
    def save_support_interaction(self, event: AfterInvocationEvent):
        """Save support interaction using MemorySession (cleaner API)"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # Get last customer query and agent response
                customer_query = None
                agent_response = None
                
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not customer_query and "toolResult" not in msg["content"][0]:
                        customer_query = msg["content"][0]["text"]
                        break
                
                if customer_query and agent_response:
                    # Use MemorySession (no need to pass actor_id/session_id)
                    interaction_messages = [
                        ConversationalMessage(customer_query, USER),
                        ConversationalMessage(agent_response, ASSISTANT)
                    ]
                    
                    result = self.customer_session.add_turns(interaction_messages)
                    logger.info(f"✅ Saved interaction using MemorySession - Event ID: {result['eventId']}")
                    
        except Exception as e:
            logger.error(f"Failed to save support interaction: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        """Register customer support memory hooks"""
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)  # Re-added!
        registry.add_callback(AfterInvocationEvent, self.save_support_interaction)
        logger.info("✅ Customer support memory hooks registered with MemorySession")
print('Executed!')

### Passo 6: Criar Agente de Suporte ao Cliente

In [ ]:
# Create memory hooks using MemorySession
support_hooks = CustomerSupportMemoryHooks(customer_session)

# Create customer support agent
support_agent = Agent(
    hooks=[support_hooks],
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[web_search, check_order_status],
    system_prompt="""You are a helpful customer support agent with access to customer history and order information. 
    
    Your role:
    - Help customers with their orders, returns, and product issues
    - Use customer context to provide personalized support
    - Search for product information when needed
    - Be empathetic and solution-focused
    - Reference previous orders and preferences when relevant
    
    Always be professional, helpful, and aim to resolve customer issues efficiently."""
)

print("✅ Customer support agent created with MemorySession integration")

### Passo 7: Alimentar Histórico do Cliente

Vamos adicionar algumas interações anteriores do cliente para demonstrar a funcionalidade de memória.

**NOTA: Esta seção usa o formato ConversationalMessage e armazenamento baseado em sessão.**

In [ ]:
# Seed with previous customer interactions using MemorySession
previous_interactions = [
    ConversationalMessage("I bought a new iPhone 15 Pro on June 1st, 2025. Order number is 123456.", USER),
    ConversationalMessage("Thank you for your purchase! I can see your iPhone 15 Pro order #123456 was delivered successfully. How can I help you today?", ASSISTANT),
    ConversationalMessage("I also ordered Sennheiser headphones on June 20th. Order number 654321. They came with 1-year warranty.", USER),
    ConversationalMessage("Perfect! I have your Sennheiser headphones order #654321 on file with the 1-year warranty. Both your iPhone and headphones should work great together.", ASSISTANT),
    ConversationalMessage("I'm looking for a good laptop. I prefer ThinkPad models.", USER),
    ConversationalMessage("Great choice! ThinkPads are excellent for their durability and performance. Let me help you find the right model for your needs.", ASSISTANT)
]

# Save using MemorySession 
try:
    event_response = customer_session.add_turns(previous_interactions)
    logger.info(f"✅ Seeded customer history using MemorySession")
    logger.info(f"   Event ID: {event_response['eventId']}")
except Exception as e:
    logger.error(f"⚠️ Error seeding history: {e}")


#### O agente está pronto. 

### Vamos testar os Cenários de Suporte ao Cliente

In [ ]:
# Test 1: Customer reports iPhone issue 
logger.info("🧪 Running Test 1: iPhone performance issue")
test_query_1 = "My iPhone is running very slow and gets hot when charging. Can you help?"
logger.info(f"Query: {test_query_1}")

response1 = support_agent(test_query_1)
logger.info(f"✅ Test 1 completed successfully")
print(f"\n📱 iPhone Issue Support Response:\n{response1}\n")

In [ ]:
# Test 2: Bluetooth connectivity issue 
logger.info("🧪 Running Test 2: Bluetooth connectivity issue")
test_query_2 = "My iPhone won't connect to my Sennheiser headphones via Bluetooth. How do I fix this?"
logger.info(f"Query: {test_query_2}")

response2 = support_agent(test_query_2)
logger.info(f"✅ Test 2 completed successfully")
print(f"\n🎧 Bluetooth Issue Support Response:\n{response2}\n")

In [ ]:
# Test 3: Check order status 
logger.info("🧪 Running Test 3: Order status check")
test_query_3 = "Can you check the status of my recent orders?"
logger.info(f"Query: {test_query_3}")

response3 = support_agent(test_query_3)
logger.info(f"✅ Test 3 completed successfully")
print(f"\n📦 Order Status Support Response:\n{response3}\n")

In [ ]:
# Test 4: Product recommendation based on preferences 
logger.info("🧪 Running Test 4: Product recommendation")
test_query_4 = "I'm still interested in buying a laptop. What ThinkPad models do you recommend?"
logger.info(f"Query: {test_query_4}")

response4 = support_agent(test_query_4)
logger.info(f"✅ Test 4 completed successfully")
print(f"\n💻 Product Recommendation Support Response:\n{response4}\n")

logger.info("🎉 All customer support scenario tests completed!")

## Recursos Avançados: Ramificação e Metadados

### Ramificação de Conversas com SessionManager

Explore cenários alternativos de suporte usando ramificação:

In [ ]:
# Get the last event ID from our conversation
events = customer_session.list_events()
if events:
    last_event_id = events[-1].eventId
    
    # Fork conversation to explore premium support path
    branch_event = customer_session.fork_conversation(
        root_event_id=last_event_id,
        branch_name="premium-support",
        messages=[
            ConversationalMessage(
                "I'd like to upgrade to premium support for faster resolution.",
                USER
            ),
            ConversationalMessage(
                "Excellent choice! With premium support, you'll get 24/7 priority assistance, dedicated account manager, and same-day resolution guarantee. Let me process your upgrade.",
                ASSISTANT
            )
        ]
    )
    
    logger.info(f"✅ Created premium support branch from event {last_event_id}")
    
    # List all branches
    branches = customer_session.list_branches()
    print(f"\n🌳 Support session has {len(branches)} branch(es):")
    for branch in branches:
        print(f"   - {branch.name}: {branch.event_count} events")
else:
    print("No events found to branch from")

### Metadados para Rastreamento Avançado de Suporte

Use metadados para rastrear métricas abrangentes de suporte:

In [ ]:
from bedrock_agentcore.memory.models import StringValue

# Add a support interaction with comprehensive metadata
metadata_event = customer_session.add_turns(
    messages=[
        ConversationalMessage(
            "The ThinkPad X1 Carbon you recommended is perfect! I'll order it now.",
            USER
        ),
        ConversationalMessage(
            "Fantastic! The ThinkPad X1 Carbon is an excellent choice for your needs. I'll help you complete the order with your preferred configuration.",
            ASSISTANT
        )
    ],
    metadata={
        "interaction_type": StringValue.build("product_recommendation"),
        "outcome": StringValue.build("purchase_intent"),
        "product_category": StringValue.build("laptops"),
        "product_brand": StringValue.build("lenovo"),
        "customer_sentiment": StringValue.build("positive"),
        "support_tier": StringValue.build("standard"),
        "session_duration_minutes": StringValue.build("15")
    }
)

logger.info(f"✅ Added support event with metadata - Event ID: {metadata_event['eventId']}")
print("\n📊 Support interaction tagged with:")
print("   - Interaction Type: product_recommendation")
print("   - Outcome: purchase_intent")
print("   - Product Category: laptops")
print("   - Customer Sentiment: positive")

### Consultas Avançadas de Metadados

Analise padrões de suporte e comportamento do cliente:

In [ ]:
from bedrock_agentcore.memory.models import EventMetadataFilter, LeftExpression, RightExpression, OperatorType

try:
    # Query product recommendation interactions
    recommendation_events = customer_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "interaction_type"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "product_recommendation"}}
            }
        ]
    )
    
    print(f"\n🛍️ Found {len(recommendation_events)} product recommendation interaction(s)")
    
    # Query positive sentiment interactions
    positive_events = customer_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "customer_sentiment"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "positive"}}
            }
        ]
    )
    
    print(f"😊 Found {len(positive_events)} positive sentiment interaction(s)")
    
    print("\n💡 Advanced analytics use cases:")
    print("   - Track conversion rates from recommendations to purchases")
    print("   - Analyze customer sentiment trends over time")
    print("   - Identify most effective support interaction types")
    print("   - Measure support tier performance")
    print("   - Generate detailed customer journey reports")
    print("   - Optimize product recommendation strategies")
    
except Exception as e:
    logger.error(f"Error querying metadata: {e}")
    print(f"Note: Metadata filtering requires events with metadata tags")

#### Tutorial de Suporte ao Cliente concluído! 🎉 
Principais aprendizados:
- Hooks de memória gerenciam automaticamente o contexto do cliente entre sessões de suporte usando MemorySessionManager
- Memória multi-estratégia captura pedidos, preferências e fatos das conversas usando classes de estratégia tipadas
- Agentes podem fornecer suporte personalizado baseado no histórico do cliente
- Ferramentas podem ser integradas para consulta de pedidos e funcionalidade de busca na web
- O suporte ao cliente se torna mais eficiente com memória persistente
- **Ramificação permite testar abordagens alternativas de suporte e caminhos de escalonamento**
- **Metadados fornecem análises abrangentes de suporte e rastreamento da jornada do cliente**

## Limpeza

### Opcional: Excluir Recurso de Memória

In [ ]:
# Clean up all resources created during notebook execution
print("=== Limpeza de Recursos ===")

# Step 1: Delete all events from the customer session (main branch)
try:
    events = customer_session.list_events()
    if events:
        logger.info(f"🗑️ Deleting {len(events)} events from main branch...")
        for event in events:
            event_id = event.eventId if hasattr(event, 'eventId') else event.get('eventId', event.get('id'))
            if event_id:
                customer_session.delete_event(event_id)
        logger.info("✅ Main branch events deleted")
    else:
        logger.info("ℹ️ No events found in main branch.")
except Exception as e:
    logger.warning(f"⚠️ Could not delete main branch events: {e}")

# Step 2: Delete events from the premium-support branch (if created)
try:
    branch_events = customer_session.list_events(branch_name="premium-support")
    if branch_events:
        logger.info(f"🗑️ Deleting {len(branch_events)} events from premium-support branch...")
        for event in branch_events:
            event_id = event.eventId if hasattr(event, 'eventId') else event.get('eventId', event.get('id'))
            if event_id:
                customer_session.delete_event(event_id, branch_name="premium-support")
        logger.info("✅ Premium-support branch events deleted")
except Exception as e:
    logger.warning(f"⚠️ Could not delete branch events: {e}")

# Step 3: Delete the Memory resource and wait for completion
try:
    logger.info(f"🗑️ Deleting memory resource: {memory_id}")
    memory_manager.delete_memory_and_wait(memory_id)
    logger.info(f"✅ Memory resource fully deleted: {memory_id}")
except Exception as e:
    logger.error(f"❌ Failed to delete memory: {e}")

# Step 4: Delete the IAM role created for custom strategies
try:
    iam_client = boto3.client('iam', region_name=REGION)
    role_name = "AgentCoreMemoryExecutionRole"
    
    # First delete the inline policy
    try:
        iam_client.delete_role_policy(
            RoleName=role_name,
            PolicyName="AgentCoreMemoryBedrockAccess"
        )
        logger.info(f"✅ Deleted inline policy from role {role_name}")
    except ClientError as e:
        if e.response['Error']['Code'] != 'NoSuchEntity':
            raise
    
    # Then delete the role
    iam_client.delete_role(RoleName=role_name)
    logger.info(f"✅ Deleted IAM role: {role_name}")
except ClientError as e:
    if e.response['Error']['Code'] == 'NoSuchEntity':
        logger.info(f"ℹ️ IAM role {role_name} already deleted or does not exist.")
    else:
        logger.error(f"❌ Failed to delete IAM role: {e}")
except Exception as e:
    logger.error(f"❌ Failed to delete IAM role: {e}")

print("=== Limpeza concluída ===")